In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [13]:
if is_main:
    seed = 43
    environment_string = "cart_pole"
    gold_timesteps = 4_000_000
    training_timesteps = 500_000
    num_concepts_selected = 11
    out_folder = "basic"
    method = "lp" 


In [27]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

In [28]:
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Optional but recommended for determinism
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
model_params = get_model(environment_string,"MlpPolicy",custom_name="cart_pole_extension")
model_params['env'] = environment_string
ground_truth_env.seed(seed)


name = "cart_pole_extension"
groundtruth_model.env = ground_truth_env

In [29]:
wandb.init(
    project="Concept Decisions",
    name=name,
    config=model_params
)
groundtruth_model.learn(total_timesteps=100_000, callback=WandbLoggingCallback())
wandb.finish()

AssertionError: 

In [10]:
evaluate_model(environment_string,ground_truth_gym_env,groundtruth_model,seed)

21.0 True True 1637.0
21.0 True True 1637.0
21.0 True True 1637.0
21.0 True True 1637.0
19.0 True True 1771.0
18.0 True True 1812.0
17.0 True True 1853.0
-1.0 True True 2414.0
19.0 True True 1742.0
19.0 True True 1768.0
19.0 True True 1790.0
18.0 True True 1839.0
14.0 True True 1981.0
17.0 True True 1811.0
18.0 True True 1806.0
18.0 True True 1802.0
21.0 True True 1637.0
21.0 True True 1637.0
18.0 True True 1778.0
18.0 True True 1778.0
18.0 True True 1877.0
12.0 True True 2033.0
21.0 True True 1637.0
2.0 True True 2379.0
19.0 True True 1790.0
16.0 True True 1849.0
-14.0 True True 1339.0
18.0 True True 1836.0
19.0 True True 1772.0
21.0 True True 1637.0
21.0 True True 1637.0
13.0 True True 2144.0
21.0 True True 1637.0
21.0 True True 1637.0
18.0 True True 1789.0
19.0 True True 1768.0
19.0 True True 1770.0
21.0 True True 1637.0
11.0 True True 2168.0
15.0 True True 2036.0
21.0 True True 1637.0
21.0 True True 1637.0
21.0 True True 1637.0
18.0 True True 1819.0
-6.0 True True 2119.0
18.0 True 

16.64

In [8]:
subset_concept, idx = random_selection(concept_list,num_concepts_selected)

In [9]:
training_timesteps = 100_000


In [13]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,subset_concept,seed,processed_concepts=processed_concepts,concept_idx=idx)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_perfect_{}_{}".format(environment_string,method,seed),seed=seed)    

approx_kl,▁▁▁▁▁▁▁▁▃▃▂▂▂▁▁▁▁▃▃▃▃▃▂▂▂▂▂▆▆▃▃▅▅▅▅▆▆▆▆█
clip_fraction,▁▁▁▁▁▁▁▁▁▁▄▄▄▁▁▁▁▁▁▁▁▁▁▁▁▆▆▁▅▃▃▃▃▃██████
ema_norm_reward,▂▁▁▂▃▂▂▄▃▂▂▂▂▂▁▃▃▃▃▂▂▂▃▂▂▂▂▂▅▆▃▅▇▆▅▇▆▇██
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅███
episode_length_mean,█▄███████▃█████████████████▁██████▄██▆▅█
episode_reward_max,▁▁▁▁▅▁▁▁▁▇▁▁▁▃▂▄▁▁▁▁▁▇▁▁▁▄▁▁▁▁▇▄▁▄▁▇▁▁█▁
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁█▁▁▁▁▂▁▁▄▁▁▃▂▃
episode_reward_min,▁▁▁▁▇▄▁▂▁▁▁▁▁▁▁▁▁▁▂▁▁▁▂▂▁▁▂▆▁▁▃▇▁▇█▁▁▂█▁
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇████
explained_variance,▁▁▁███████▇▇▇▆▆▆▅▅▅▅▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅
+1,...


In [11]:
model_name = "../../results/models/concept_predictor_env={}_training={}_seed={}.pth".format(environment_string,100,seed)

height = width = 84

if environment_string == "mini_grid":
    num_frames = 1
else:
    num_frames = 4

if environment_string == "cart_pole":
    height = 160
    width = 240

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if os.path.exists(model_name):
    concept_predictor = ConceptPredictorCNN(len(concept_list), num_frames=num_frames,height=height,width=width).to(device)
    concept_predictor.load_state_dict(torch.load(model_name, weights_only=True))
    concept_predictor.eval()

In [12]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,fast_predictor=concept_predictor,use_processed=True,concept_idx=idx,processed_concepts=processed_concepts)
model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=training_timesteps,custom_name="{}_imperfect_{}_{}".format(environment_string,method,seed)) 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  

approx_kl,▅▅▅▅▂▇▇▇▇▇▅▅▅▅▁▁▁▄▄▄▄▅▅▅▃▃▃███▂▂▂▂▄▄▄▃▃▃
clip_fraction,▄▄▄▄▁▃▃▃▃▃▄▄▁▁▁▁▁▁▁▁▃▃▃▁██████▁▁▁▁▁▁▁▁▁▁
ema_norm_reward,▂▃▂▂▂▁▁▂▁▁▁▁▁▁▁▃▂▂▂▂▃▁▂▂▂▂▂▂▂▁▄▆▅▆▅▄▄▆▃█
entropy_loss,▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███
episode_length_mean,█▇███████████████████▇▇▁███▂█▄█▅▆██▅█▃▅▅
episode_reward_max,▁▁▁▁▁▁▂▁▁▁▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▆▅▁▄▄▃▁▃▃▁█▁▁
episode_reward_mean,█▃▁▁▁▁▁▁▁▁▁▃▁▂▃▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▂▅▃█▄▁▁
episode_reward_min,▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▄▁▁▁▁▁▁▃▁▄▁▁▁▄▁▁▁▁▅
episodes_completed,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇██████
explained_variance,▁▁▁▁▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇█████████
+1,...
